# GéoMarketing IDF : J13e — Diagnostic de marché, comparaison territoriale et exports pour Streamlit


## Objectifs
- Croiser population, profils sociaux, revenus disponibles, restauration, équipements et transports.
- Comparer les communes et les IRIS à leur maille réelle.
- Conserver tous les IRIS d’Île-de-France et produire un focus Cergy–Pontoise.
- Expliquer les indicateurs et leur disponibilité à un client non technique.
- Préparer des tables légères et des contours pour la première application.
## Périmètre
Toute l’IDF pour les tables, Cergy (95127) et Pontoise (95500) pour les cartes locales
et les fiches de lecture. Géographie source par défaut : 2024 ; RP : 2022.
Les observations BPE/IDFM peuvent avoir d’autres dates, conservées dans la provenance.
## Interprétation
Le diagnostic décrit et compare les territoires. Une densité faible ne démontre
pas un besoin commercial non satisfait ; une densité forte ne démontre pas une
saturation. Les effectifs travail/études de J13d sont communaux et ne sont pas
répartis entre les IRIS. Les médianes fiscales restent des données IRIS publiées.
Aucun chiffre d’affaires prévisionnel, score de rentabilité ou total de clientèle
unique n’est calculé. Un emplacement précis demande ensuite une étude terrain.

## 1. Imports et environnement

In [1]:
import os
os.environ['OMP_NUM_THREADS']='1'
os.environ['NUMEXPR_NUM_THREADS']='1'
os.environ['MKL_NUM_THREADS']='1'
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import csv, json, re, hashlib, html
import numpy as np
import pandas as pd
import geopandas as gpd
import plotly
import plotly.graph_objects as go
from IPython.display import display
pd.set_option('display.max_columns',80)
pd.set_option('display.max_rows',30)
print('pandas',pd.__version__,'; geopandas',gpd.__version__,'; plotly',plotly.__version__)


pandas 2.2.2 ; geopandas 1.0.1 ; plotly 5.24.1


## 2 — Chemins des fichiers à reprendre
Aucun téléchargement brut supplémentaire n’est nécessaire. Les entrées sont les
sorties contrôlées J13c/J13d, pas les fichiers nationaux. Adapter les noms si vos
sorties locales portent d’autres noms. Le CSV contient les attributs ; le GPKG
fournit les contours complets. Les dictionnaires antérieurs sont facultatifs.

In [2]:
RACINE=Path(os.environ.get('GEOMARKETING_RACINE',r'C:\Users\almou\OneDrive\GeoMarketing_IDF'))
P13C=RACINE/'data/processed/j13c'
P13D=RACINE/'data/processed/j13d'
FICHIER_IRIS=P13D/'profil_iris_idf_j13d.csv'
FICHIER_COMMUNES=P13D/'profil_communes_mobilites_idf_j13d.csv'
FICHIER_GEO=P13D/'profil_iris_idf_j13d.gpkg'
COUCHE_IRIS='profils_iris'
FICHIER_POINTS=P13C/'points_affectations_idf_j13c.csv'
FICHIER_CATEGORIES=P13C/'categories_points_par_iris_j13c.csv'
FICHIER_META_C=P13C/'metadata_affectations_iris_j13c.json'
FICHIER_META_D=P13D/'metadata_mobilites_j13d.json'
FICHIERS_DICTIONNAIRES=[RACINE/'docs/dictionnaire_j13b.csv',
                       RACINE/'docs/dictionnaire_j13c.csv',RACINE/'docs/dictionnaire_j13d.csv']
DOSSIER_INTERIM=RACINE/'data/interim/j13e'
DOSSIER_PROCESSED=RACINE/'data/processed/j13e'
DOSSIER_APP=RACINE/'data/app/j13e'
DOSSIER_CARTES=RACINE/'outputs/cartes/j13e'
DOSSIER_RAPPORTS=RACINE/'outputs/rapports/j13e'
DOSSIER_DOCS=RACINE/'docs'
for dossier in [DOSSIER_INTERIM,DOSSIER_PROCESSED,DOSSIER_APP,DOSSIER_CARTES,DOSSIER_RAPPORTS,DOSSIER_DOCS]:
    dossier.mkdir(parents=True,exist_ok=True)
ENTREES=[FICHIER_IRIS,FICHIER_COMMUNES,FICHIER_GEO,FICHIER_POINTS,
         FICHIER_CATEGORIES,FICHIER_META_C,FICHIER_META_D]
print('Racine :',RACINE)
display(pd.DataFrame([{'FICHIER':str(p),'EXISTE':p.is_file()} for p in ENTREES]))


Racine : C:\Users\almou\OneDrive\GeoMarketing_IDF


,FICHIER,EXISTE
0,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\...,True
1,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\...,True
2,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\...,True
3,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\...,True
4,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\...,True
5,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\...,True
6,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\...,True


## 3 — Paramètres de comparaison
Les seuils ci-dessous sont des choix de présentation, pas des seuils de viabilité.
La référence IRIS est calculée séparément par type (habitat, activité, divers,
commune non découpée, etc.). Cela évite de classer ensemble des mailles différentes.
Le focus ne supprime aucune ligne des tables régionales. La marge cartographique
ne représente ni une zone de chalandise ni une distance de marche.

In [3]:
ANNEE_RP=2022
GEOGRAPHIE=2024
DEPARTEMENTS_IDF={'75','77','78','91','92','93','94','95'}
CODES_FOCUS=['95127','95500']
MIN_POP_IRIS_COMPARAISON=500
MIN_POP_COMMUNE_COMPARAISON=5000
MIN_REFERENCE=5
TAILLE_BLOC_POINTS=25000
MARGE_CARTE_METRES=500
SIMPLIFICATION_CARTE_METRES=15
MAX_POINTS_CARTE=20000
MAX_FICHIER_APP_MO=20
AFFICHER_CARTES=False
COL_MED='J13B_NIVEAU_VIE_MEDIAN_ANNUEL_UC_EUR'
COL_PAUV='J13B_TAUX_PAUVRETE_60_PCT'
CONFIG_PLOTLY={'displaylogo':False,'scrollZoom':True}
if MIN_REFERENCE<2:raise ValueError('MIN_REFERENCE doit être au moins 2.')
if not CODES_FOCUS:raise ValueError('Renseigner au moins une commune du focus.')


## 4 — Lectures strictes, nombres et exports
Les codes restent du texte. Les CSV préparés sont attendus en UTF-8, avec une
première ligne d’en-tête. Le lecteur accepte `;`, `,` et tabulation, vérifie tous
les champs même s’ils ne sont pas utilisés, et ne saute aucune ligne mal formée.
Les booléens inconnus et nombres non interprétables produisent une erreur explicite.
Les fichiers de sortie sont distincts des entrées et écrits via un fichier temporaire.

In [4]:
def texte(s):
    return s.astype('string').str.strip().replace({'':pd.NA,'nan':pd.NA,'NaN':pd.NA,'<NA>':pd.NA})

def nombre(s,libelle='valeur'):
    t=texte(s).str.replace('\u202f','',regex=False).str.replace('\u00a0','',regex=False)
    t=t.str.replace(' ','',regex=False).str.replace(',','.',regex=False)
    n=pd.to_numeric(t,errors='coerce')
    bad=t.notna() & (n.isna() | ~np.isfinite(n.astype(float)))
    if bad.any():raise ValueError(libelle+' : nombres invalides '+str(t[bad].head(5).tolist()))
    return n.astype(float)

def booleen(s,libelle):
    t=texte(s).str.lower()
    if not t.isin(['true','false','1','0']).all():raise ValueError(libelle+' : booléen absent/inconnu.')
    return t.isin(['true','1'])

def entete_csv(p):
    with Path(p).open('r',encoding='utf-8-sig',newline='') as f:ligne=f.readline()
    candidats=[]
    for sep in [';',',','\t']:
        cols=next(csv.reader([ligne],delimiter=sep,strict=True))
        if len(cols)>1:candidats.append((sep,cols))
    if len(candidats)!=1:raise ValueError(str(p)+' : séparateur/en-tête ambigu ou une seule colonne.')
    sep,cols=candidats[0]
    if len(cols)!=len(set(cols)) or any(not c.strip() for c in cols):
        raise ValueError(str(p)+' : en-têtes dupliqués ou vides.')
    return sep,cols

def blocs_csv(p,usecols=None,taille=25000):
    sep,cols=entete_csv(p)
    if usecols is None:usecols=cols
    manque=set(usecols)-set(cols)
    if manque:raise ValueError(Path(p).name+' : colonnes absentes '+str(sorted(manque)))
    ix=[cols.index(c) for c in usecols]
    with Path(p).open('r',encoding='utf-8-sig',newline='') as f:
        lecteur=csv.reader(f,delimiter=sep,strict=True)
        next(lecteur);lignes=[]
        try:
            for ligne in lecteur:
                if not ligne:continue
                if len(ligne)!=len(cols):
                    raise ValueError(Path(p).name+' : ligne '+str(lecteur.line_num)+
                                     ', '+str(len(ligne))+' champs au lieu de '+str(len(cols))+'.')
                lignes.append([ligne[i] for i in ix])
                if len(lignes)>=taille:
                    yield pd.DataFrame(lignes,columns=usecols,dtype='string').replace('',pd.NA);lignes=[]
            if lignes:yield pd.DataFrame(lignes,columns=usecols,dtype='string').replace('',pd.NA)
        except csv.Error as exc:raise ValueError(Path(p).name+' : CSV mal formé.') from exc

def lire_csv(p):
    cols=entete_csv(p)[1];tables=list(blocs_csv(p))
    return pd.concat(tables,ignore_index=True) if tables else pd.DataFrame(columns=cols,dtype='string')

def empreinte(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()

SORTIES=[]
def exporter_csv(t,p,sep=','):
    p=Path(p)
    if p.resolve() in {x.resolve() for x in ENTREES}:raise ValueError('Une sortie écraserait une entrée.')
    temp=p.with_name(p.stem+'_temp'+p.suffix)
    t.to_csv(temp,index=False,sep=sep,encoding='utf-8-sig')
    temp.replace(p)
    if p not in SORTIES:SORTIES.append(p)
    return p

def exporter_json(obj,p):
    p=Path(p);temp=p.with_name(p.stem+'_temp'+p.suffix)
    temp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
    temp.replace(p)
    if p not in SORTIES:SORTIES.append(p)
    return p

def ratio(a,b,facteur=100):return a.div(b.where(b.gt(0)))*facteur
def somme_complete(s):return s.sum() if s.notna().all() else np.nan
def serie_numerique(t,col):
    return nombre(t[col],col) if col in t else pd.Series(np.nan,index=t.index,dtype=float)


## 5 — Inventaire et métadonnées
Ces métadonnées assurent la continuité J13c/J13d. Le millésime des observations
BPE ou IDFM n’est pas confondu avec la géographie des IRIS. Les empreintes des
entrées seront enregistrées à la fin du notebook, sans inventer leurs dates.

In [5]:
for p in ENTREES:
    if not p.is_file():raise FileNotFoundError('Sortie amont manquante ; terminer J13c/J13d ou adapter le bloc 2 : '+str(p))
meta_c=json.loads(FICHIER_META_C.read_text(encoding='utf-8'))
meta_d=json.loads(FICHIER_META_D.read_text(encoding='utf-8'))
if (meta_c.get('annee_rp'),meta_c.get('geographie_iris'))!=(ANNEE_RP,GEOGRAPHIE):
    raise ValueError('J13c : RP/géographie incompatibles avec le bloc 3.')
if (meta_d.get('annee_rp'),meta_d.get('geographie'))!=(ANNEE_RP,GEOGRAPHIE):
    raise ValueError('J13d : RP/géographie incompatibles avec le bloc 3.')
if meta_d.get('maille_flux')!='COMMUNE':raise ValueError('J13d : maille des mobilités non reconnue.')
inventaire=pd.DataFrame([{'FICHIER':str(p.relative_to(RACINE)) if p.is_relative_to(RACINE) else str(p),
                         'TAILLE_MO':p.stat().st_size/1_000_000} for p in ENTREES])
exporter_csv(inventaire,DOSSIER_INTERIM/'inventaire_entrees_j13e.csv')
display(inventaire)
display(pd.DataFrame(meta_c.get('lectures',[])))


,FICHIER,TAILLE_MO
0,data\processed\j13d\profil_iris_idf_j13d.csv,17.506001
1,data\processed\j13d\profil_communes_mobilites_...,0.503455
2,data\processed\j13d\profil_iris_idf_j13d.gpkg,25.825280
3,data\processed\j13c\points_affectations_idf_j1...,215.081897
4,data\processed\j13c\categories_points_par_iris...,16.795328
5,data\processed\j13c\metadata_affectations_iris...,0.014603
6,data\processed\j13d\metadata_mobilites_j13d.json,0.005586


,SOURCE,FICHIER,COL_ID,COL_NOM,COL_CATEGORIE,COL_X,COL_Y,COL_COORDONNEES,COL_COMMUNE,CRS_TRAVAIL,CRS_SOURCE,ANNEE_SOURCE,DATE_SOURCE,UNITE
0,RESTAURATION,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\...,ID_EQUIPEMENT,NOM_ETABLISSEMENT,GROUPE_EQUIPEMENT,X_2154,Y_2154,None,DEPCOM,2154,2154,2025.0,None,"Observation de restaurant BPE, dédupliquée sel..."
1,TRANSPORTS,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\...,ID_EQUIPEMENT,NOM_ETABLISSEMENT,GROUPE_EQUIPEMENT,X_2154,Y_2154,None,DEPCOM,2154,2154,NaN,None,Point de transport selon l’identifiant et la g...
2,EQUIPEMENTS,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\...,ID_EQUIPEMENT,NOM_ETABLISSEMENT,GROUPE_EQUIPEMENT,X_2154,Y_2154,None,DEPCOM,2154,2154,2025.0,None,Observation d’équipement complémentaire de la ...


## 6 — Charger les profils et contrôler leurs clés
Le profil complet garde toutes les colonnes héritées. Les nouveautés commencent
par J13E_. La clé de contexte communal est celle de J13d, qui respecte la
géographie source : Paris regroupé, Pierrefitte distincte en géographie 2024.
CODGEO_ETUDE ne remplace pas implicitement cette clé.

In [6]:
base=lire_csv(FICHIER_IRIS)
communes_amont=lire_csv(FICHIER_COMMUNES)
requis={'IRIS','CODGEO_SOURCE','CODGEO_ETUDE','NOM_IRIS','J13A_POPULATION',
        'J13A_ANNEE_RP','J13A_GEOGRAPHIE','J13D_COMMUNE_CONTEXTE'}
if requis-set(base):raise ValueError('Profil IRIS incomplet : '+str(sorted(requis-set(base))))
if any(c.startswith('J13E_') for c in base):raise ValueError('Choisir une entrée J13d, pas une sortie J13e.')
if not base['IRIS'].str.fullmatch(r'\d{9}',na=False).all() or not base['IRIS'].is_unique:
    raise ValueError('Codes IRIS invalides ou dupliqués.')
if not base['CODGEO_SOURCE'].eq(base['IRIS'].str[:5]).all():raise ValueError('IRIS / commune source incohérents.')
if set(base['IRIS'].str[:2])!=DEPARTEMENTS_IDF:raise ValueError('Les huit départements IDF doivent être conservés.')
for col,attendu in [('J13A_ANNEE_RP',ANNEE_RP),('J13A_GEOGRAPHIE',GEOGRAPHIE)]:
    if not nombre(base[col],col).eq(attendu).all():raise ValueError(col+' incompatible.')
for col in ['CODGEO','NOM_COMMUNE','J13D_ANNEE_RP','J13D_GEOGRAPHIE']:
    if col not in communes_amont:raise ValueError('Communes J13d : colonne absente '+col)
if not communes_amont['CODGEO'].str.fullmatch(r'\d{5}',na=False).all() or not communes_amont['CODGEO'].is_unique:
    raise ValueError('Codes communaux invalides ou dupliqués.')
if not nombre(communes_amont['J13D_ANNEE_RP']).eq(ANNEE_RP).all() or not nombre(communes_amont['J13D_GEOGRAPHIE']).eq(GEOGRAPHIE).all():
    raise ValueError('Communes J13d : millésime ou géographie incompatible.')
if set(base['J13D_COMMUNE_CONTEXTE'])!=set(communes_amont['CODGEO']):raise ValueError('Couverture IRIS/communes différente.')
FOCUS=set(CODES_FOCUS)
if FOCUS-set(communes_amont['CODGEO']):raise ValueError('Commune du focus absente.')
LIBELLE_FOCUS=' et '.join(communes_amont.set_index('CODGEO').loc[CODES_FOCUS,'NOM_COMMUNE'].astype(str))
ATTENDUS=set(base['IRIS']);NOMBRE_IRIS=len(base)
profil_iris=base.copy()
profil_iris['J13E_DANS_FOCUS']=base['J13D_COMMUNE_CONTEXTE'].isin(FOCUS)
profil_iris['J13E_NOM_COMMUNE_CONTEXTE']=base['J13D_COMMUNE_CONTEXTE'].map(communes_amont.set_index('CODGEO')['NOM_COMMUNE'])
profil_iris['J13E_TYPE_IRIS_COMPARAISON']=texte(base['TYP_IRIS']).fillna('NON_RENSEIGNE') if 'TYP_IRIS' in base else 'NON_RENSEIGNE'
print('IRIS :',NOMBRE_IRIS,'; communes :',len(communes_amont),' ; IRIS focus :',int(profil_iris['J13E_DANS_FOCUS'].sum()))


IRIS : 5264 ; communes : 1267  ; IRIS focus : 37


## 7 — Contours complets et communes
Les surfaces sont calculées en Lambert-93 sur les contours complets. Seuls les
contours destinés aux cartes et à l’application seront simplifiés. Les géométries
ne permettent pas d’attribuer les flux communaux à des quartiers.

In [7]:
iris_geo=gpd.read_file(FICHIER_GEO,layer=COUCHE_IRIS)
if iris_geo.crs is None or 'IRIS' not in iris_geo:raise ValueError('Contours : IRIS et CRS requis.')
iris_geo['IRIS']=iris_geo['IRIS'].astype('string')
if not iris_geo['IRIS'].is_unique or set(iris_geo['IRIS'])!=ATTENDUS:raise ValueError('Clés CSV/GPKG différentes.')
if iris_geo.geometry.isna().any() or iris_geo.geometry.is_empty.any() or not iris_geo.geometry.is_valid.all():
    raise ValueError('Contours vides ou invalides ; corriger en amont.')
if not iris_geo.geom_type.isin(['Polygon','MultiPolygon']).all():raise ValueError('Polygones IRIS attendus.')
iris_geo=iris_geo[['IRIS',iris_geo.geometry.name]].to_crs(2154)
if iris_geo.geometry.name!='geometry':iris_geo=iris_geo.rename_geometry('geometry')
geo_communes=iris_geo.merge(base[['IRIS','J13D_COMMUNE_CONTEXTE']],on='IRIS',validate='one_to_one')
geo_communes=geo_communes.rename(columns={'J13D_COMMUNE_CONTEXTE':'CODGEO'}).dissolve(by='CODGEO',as_index=False)[['CODGEO','geometry']]
profil_iris['J13E_SURFACE_KM2']=profil_iris['IRIS'].map(iris_geo.set_index('IRIS').area/1_000_000)
if not profil_iris['J13E_SURFACE_KM2'].gt(0).all():raise ValueError('Surface IRIS non positive.')


## 8 — Contrôler les sources et les valeurs manquantes
Une source non chargée donne des valeurs manquantes. Dans une source chargée,
zéro signifie aucune observation retenue dans cette maille, sans garantir une
absence réelle. Les comptes doivent correspondre aux points de J13c. Une source
partiellement chargée au niveau régional n’est pas reconstruite automatiquement.

In [8]:
NOMS_COMPTES={'RESTAURATION':'J13C_NB_OBS_RESTAURATION','TRANSPORTS':'J13C_NB_POINTS_ARRET',
              'EQUIPEMENTS':'J13C_NB_OBS_EQUIPEMENTS'}
SOURCES_CHARGEES={}
for source,col in NOMS_COMPTES.items():
    flag='J13C_SOURCE_'+source+'_CHARGEE'
    if flag not in base or col not in base:raise ValueError('Profil J13c incomplet : '+flag+' / '+col)
    b=booleen(base[flag],flag)
    if b.nunique()!=1:raise ValueError(source+' : disponibilité non uniforme ; contrôler J13c.')
    SOURCES_CHARGEES[source]=bool(b.iloc[0])
    n=nombre(base[col],col)
    if n.lt(0).any() or (n.dropna()%1!=0).any():raise ValueError(col+' : compte négatif/non entier.')
    if b.iloc[0] and n.isna().any():raise ValueError(col+' : compte manquant malgré source chargée.')
    if not b.iloc[0] and n.notna().any():raise ValueError(col+' : valeurs présentes malgré source non chargée.')
rapid=serie_numerique(base,'J13C_NB_OBS_RESTAURATION_RAPIDE')
total=serie_numerique(base,NOMS_COMPTES['RESTAURATION'])
if rapid.lt(0).any() or rapid.gt(total).any():raise ValueError('Comptes de restauration rapide incohérents.')
if SOURCES_CHARGEES['RESTAURATION'] and (rapid.isna().any() or (rapid.dropna()%1!=0).any()):
    raise ValueError('Restauration rapide : compte manquant ou non entier.')
if not SOURCES_CHARGEES['RESTAURATION'] and rapid.notna().any():raise ValueError('Source restauration non chargée.')
for col in ['J13A_POPULATION','J13A_POP_1529','J13B_NB_SCOLARISES_1824',COL_MED,COL_PAUV]:
    n=serie_numerique(base,col)
    if n.lt(0).any():raise ValueError(col+' : valeur négative.')
if serie_numerique(base,COL_PAUV).gt(100).any():raise ValueError('Taux de pauvreté supérieur à 100 %.')
for theme in ['MOBPRO','MOBSCO']:
    flag='J13D_'+theme+'_CHARGE'
    if flag not in communes_amont:raise ValueError('Communes J13d : colonne absente '+flag)
    charge=booleen(communes_amont[flag],flag)
    if charge.nunique()!=1:raise ValueError(theme+' : disponibilité régionale non uniforme.')
    if bool(charge.iloc[0])!=(theme in {s.get('theme') for s in meta_d.get('sources',[])}):
        raise ValueError(theme+' : disponibilité différente des métadonnées J13d.')
    mesures=['RESIDENTS','INTERNES','ENTRANTS','SORTANTS_FRANCE','SORTANTS_ETRANGER','DEST_NON_LOCALISEES','LIEU']
    for mesure in mesures:
        col='J13D_'+theme+'_'+mesure
        if col not in communes_amont:raise ValueError('Communes J13d : colonne absente '+col)
        n=nombre(communes_amont[col],col)
        if n.lt(0).any() or (charge.iloc[0] and n.isna().any()):raise ValueError(col+' : effectif manquant/négatif.')
    for col in [c for c in communes_amont if c.startswith('J13D_'+theme+'_') and c not in [flag,'J13D_MOBSCO_AGES_DISPONIBLES']]:
        n=nombre(communes_amont[col],col)
        if n[~charge].notna().any():raise ValueError(col+' : source désactivée avec valeurs présentes.')
display(pd.DataFrame([{'SOURCE':s,'CHARGEE':v} for s,v in SOURCES_CHARGEES.items()]))


,SOURCE,CHARGEE
0,RESTAURATION,True
1,TRANSPORTS,True
2,EQUIPEMENTS,True


## 9 — Agrégations communales sans moyenne de médianes
Les effectifs IRIS sont additionnés seulement si toutes leurs valeurs sont
disponibles. Une commune avec un IRIS de population manquante conserve un total
manquant et une couverture explicite. Les pourcentages sont recalculés sur les
sommes de numérateurs et dénominateurs. Les revenus médians et taux de pauvreté
IRIS ne sont ni sommés ni moyennés pour produire un indicateur fiscal communal.

In [9]:
communes=communes_amont.copy()
cle=base['J13D_COMMUNE_CONTEXTE']
AGREGATIONS={
 'J13E_POPULATION':'J13A_POPULATION',
 'J13E_POP_1529':'J13A_POP_1529',
 'J13E_NB_SCOLARISES_1824_RESIDENTS':'J13B_NB_SCOLARISES_1824',
 'J13E_NB_OBS_RESTAURATION':'J13C_NB_OBS_RESTAURATION',
 'J13E_NB_OBS_RESTAURATION_RAPIDE':'J13C_NB_OBS_RESTAURATION_RAPIDE',
 'J13E_NB_POINTS_ARRET':'J13C_NB_POINTS_ARRET',
 'J13E_NB_OBS_EQUIPEMENTS':'J13C_NB_OBS_EQUIPEMENTS'}
for sortie,source in AGREGATIONS.items():
    s=serie_numerique(base,source)
    somme=pd.DataFrame({'CODGEO':cle,'VALEUR':s}).groupby('CODGEO')['VALEUR'].agg(somme_complete)
    communes[sortie]=communes['CODGEO'].map(somme)
communes['J13E_NB_IRIS']=communes['CODGEO'].map(cle.value_counts())
pop_ok=serie_numerique(base,'J13A_POPULATION').notna()
couverture_pop=pop_ok.groupby(cle).sum()
communes['J13E_NB_IRIS_POPULATION_DISPONIBLE']=communes['CODGEO'].map(couverture_pop)
communes['J13E_COUVERTURE_POPULATION_IRIS_PCT']=ratio(communes['J13E_NB_IRIS_POPULATION_DISPONIBLE'],communes['J13E_NB_IRIS'])
communes['J13E_SURFACE_KM2']=communes['CODGEO'].map(geo_communes.set_index('CODGEO').area/1_000_000)
communes['J13E_DANS_FOCUS']=communes['CODGEO'].isin(FOCUS)
for source in SOURCES_CHARGEES:communes['J13E_SOURCE_'+source+'_CHARGEE']=SOURCES_CHARGEES[source]
pop_j13d=serie_numerique(communes_amont,'J13D_POPULATION_RP_IRIS')
if not np.allclose(communes['J13E_POPULATION'],pop_j13d,equal_nan=True):raise ValueError('Population agrégée différente de J13d.')


## 10 — Densités et parts recalculées
Les indicateurs sont produits aux deux mailles, avec le même libellé mais la
maille précisée dans le dictionnaire. Un IRIS d’activité peu peuplé peut avoir une
densité par habitant très forte ; cette valeur ne suffit pas à évaluer son marché.
Le ratio restauration / personnes au lieu de travail n’est calculé qu’à la commune.
Il croise des dates différentes et sert à décrire une relation, pas une clientèle.

In [10]:
profil_iris['J13E_POPULATION']=serie_numerique(base,'J13A_POPULATION')
profil_iris['J13E_POP_1529']=serie_numerique(base,'J13A_POP_1529')
profil_iris['J13E_NB_SCOLARISES_1824_RESIDENTS']=serie_numerique(base,'J13B_NB_SCOLARISES_1824')
for sortie,source in AGREGATIONS.items():
    if sortie.startswith('J13E_NB_') and sortie not in profil_iris:profil_iris[sortie]=serie_numerique(base,source)
FORMULES={
 'J13E_PART_1529_PCT':('J13E_POP_1529','J13E_POPULATION',100),
 'J13E_DENSITE_POPULATION_HAB_KM2':('J13E_POPULATION','J13E_SURFACE_KM2',1),
 'J13E_DENSITE_OBS_RESTAURATION_10000_HAB':('J13E_NB_OBS_RESTAURATION','J13E_POPULATION',10000),
 'J13E_DENSITE_OBS_RAPIDE_10000_HAB':('J13E_NB_OBS_RESTAURATION_RAPIDE','J13E_POPULATION',10000),
 'J13E_PART_RESTAURATION_RAPIDE_PCT':('J13E_NB_OBS_RESTAURATION_RAPIDE','J13E_NB_OBS_RESTAURATION',100),
 'J13E_DENSITE_POINTS_ARRET_KM2':('J13E_NB_POINTS_ARRET','J13E_SURFACE_KM2',1)}
for t in [profil_iris,communes]:
    for col,(num,den,facteur) in FORMULES.items():t[col]=ratio(t[num],t[den],facteur)
    if t['J13E_PART_1529_PCT'].gt(100+1e-6).any():raise ValueError('Population 15–29 supérieure au total.')
communes['J13E_OBS_RESTAURATION_1000_PERSONNES_LIEU_TRAVAIL']=ratio(
    communes['J13E_NB_OBS_RESTAURATION'],serie_numerique(communes,'J13D_MOBPRO_LIEU'),1000)
for t,cle_t in [(profil_iris,'IRIS'),(communes,'CODGEO')]:
    display(t.loc[t['J13E_DANS_FOCUS'],[cle_t,'J13E_POPULATION','J13E_NB_OBS_RESTAURATION',
          'J13E_DENSITE_OBS_RESTAURATION_10000_HAB','J13E_PART_RESTAURATION_RAPIDE_PCT']])


,IRIS,J13E_POPULATION,J13E_NB_OBS_RESTAURATION,J13E_DENSITE_OBS_RESTAURATION_10000_HAB,J13E_PART_RESTAURATION_RAPIDE_PCT
4703,955000304,4262.914601,5.0,11.729064,0.000000
4735,955000103,3090.766322,5.0,16.177218,0.000000
4746,951270106,3856.871645,21.0,54.448273,42.857143
4768,951270113,2244.011600,5.0,22.281525,100.000000
4774,951270127,2358.520974,2.0,8.479891,100.000000
...,...,...,...,...,...
5145,951270112,2299.352032,0.0,0.000000,NaN
5162,951270123,1526.417849,1.0,6.551286,100.000000
5185,951270117,1883.313337,1.0,5.309791,100.000000
5205,951270103,1482.589740,3.0,20.234863,66.666667


,CODGEO,J13E_POPULATION,J13E_NB_OBS_RESTAURATION,J13E_DENSITE_OBS_RESTAURATION_10000_HAB,J13E_PART_RESTAURATION_RAPIDE_PCT
1120,95127,69578.0,208.0,29.894507,66.346154
1221,95500,31623.0,128.0,40.476868,58.593750


## 11 — Revenus et couverture fiscale
Reprendre les valeurs et statuts hérités de J13b ; ce notebook ne répare pas une
jointure fiscale absente. Un revenu manquant reste manquant. La couverture
communale est le pourcentage d’IRIS renseignés, pas le pourcentage de ménages.
Une médiane disponible ne justifie pas seule un positionnement commercial premium.

In [11]:
couverture_revenus=[]
for code_commune,g in profil_iris.groupby('J13D_COMMUNE_CONTEXTE'):
    med=serie_numerique(g,COL_MED);pau=serie_numerique(g,COL_PAUV)
    couverture_revenus.append({'CODGEO':code_commune,'NB_IRIS':len(g),
        'NB_IRIS_MEDIANE_DISPONIBLE':int(med.notna().sum()),
        'NB_IRIS_PAUVRETE_DISPONIBLE':int(pau.notna().sum()),
        'COUVERTURE_MEDIANE_IRIS_PCT':100*med.notna().mean(),
        'COUVERTURE_PAUVRETE_IRIS_PCT':100*pau.notna().mean()})
for col,flag in [(COL_MED,'J13B_MEDIANE_REVENUS_DISPONIBLE'),(COL_PAUV,'J13B_TAUX_PAUVRETE_DISPONIBLE')]:
    if col in base and flag in base and not booleen(base[flag],flag).eq(serie_numerique(base,col).notna()).all():
        raise ValueError(col+' : disponibilité fiscale incohérente ; reprendre J13b.')
couverture_revenus=pd.DataFrame(couverture_revenus)
exporter_csv(couverture_revenus,DOSSIER_INTERIM/'couverture_revenus_j13e.csv')
if 'J13B_STATUT_REVENUS' in base:
    statuts=base['J13B_STATUT_REVENUS'].fillna('STATUT_ABSENT').value_counts().rename_axis('STATUT').reset_index(name='NB_IRIS')
else:statuts=pd.DataFrame([{'STATUT':'COLONNE_STATUT_ABSENTE','NB_IRIS':len(base)}])
exporter_csv(statuts,DOSSIER_INTERIM/'statuts_revenus_j13e.csv')
display(statuts)
display(couverture_revenus[couverture_revenus['CODGEO'].isin(FOCUS)])


,STATUT,NB_IRIS
0,DISPONIBLE,3950
1,GEOGRAPHIE_NON_HARMONISEE,935
2,VALEURS_NON_DIFFUSEES_OU_MANQUANTES,376
3,PARTIEL,3


,CODGEO,NB_IRIS,NB_IRIS_MEDIANE_DISPONIBLE,NB_IRIS_PAUVRETE_DISPONIBLE,COUVERTURE_MEDIANE_IRIS_PCT,COUVERTURE_PAUVRETE_IRIS_PCT
1120,95127,27,21,21,77.777778,77.777778
1221,95500,10,9,9,90.000000,90.000000


## 12 — Vérifier les points J13c par lecture progressive
Tous les points sont parcourus pour contrôler les comptes hérités ; seuls les
points du focus ou de sa marge cartographique sont gardés en mémoire. Les points
non comptés conservent leurs statuts. Le total des points d’arrêt suit exactement
l’unité choisie dans J13c, qui peut être un arrêt, une zone ou une observation par
ligne. Il ne devient pas un nombre de gares uniques.

In [12]:
colonnes_points=['SOURCE','ID_SOURCE','CLE_POINT','NOM','CATEGORIE','CATEGORIE_CODE','TYPEQU',
 'IRIS_AFFECTE','STATUT_AFFECTATION','COORDONNEES_VALIDES','INCLUS_COMPTAGE',
 'LONGITUDE','LATITUDE','X_2154','Y_2154']
iris_focus=set(profil_iris.loc[profil_iris['J13E_DANS_FOCUS'],'IRIS'])
geo_focus=iris_geo[iris_geo['IRIS'].isin(iris_focus)]
x0,y0,x1,y1=geo_focus.total_bounds
comptes_points=Counter();comptes_categories=Counter();statistiques=Counter();cles_vues=set();locaux=[]
for b in blocs_csv(FICHIER_POINTS,colonnes_points,TAILLE_BLOC_POINTS):
    if b['CLE_POINT'].isna().any() or b['CLE_POINT'].duplicated().any() or set(b['CLE_POINT'])&cles_vues:
        raise ValueError('Points J13c : clé vide/dupliquée, y compris entre blocs.')
    cles_vues.update(b['CLE_POINT'])
    if not b['SOURCE'].isin(NOMS_COMPTES).all():raise ValueError('Points : source non reconnue.')
    for c in ['COORDONNEES_VALIDES','INCLUS_COMPTAGE']:b[c]=booleen(b[c],c)
    for c in ['LONGITUDE','LATITUDE','X_2154','Y_2154']:b[c]=nombre(b[c],c)
    if (b['INCLUS_COMPTAGE'] & (~b['COORDONNEES_VALIDES']|~b['IRIS_AFFECTE'].isin(ATTENDUS))).any():
        raise ValueError('Point compté sans coordonnées/IRIS valide.')
    for source in b['SOURCE'].unique():
        if not SOURCES_CHARGEES[source]:raise ValueError('Points présents pour une source non chargée : '+source)
    ok=b[b['INCLUS_COMPTAGE']]
    for (source,iris),n in ok.groupby(['SOURCE','IRIS_AFFECTE']).size().items():comptes_points[(source,iris)]+=int(n)
    if ok['CATEGORIE_CODE'].isna().any():raise ValueError('Point compté sans catégorie.')
    for (source,iris,cat),n in ok.groupby(['SOURCE','IRIS_AFFECTE','CATEGORIE_CODE']).size().items():
        comptes_categories[(source,iris,cat)]+=int(n)
    for (source,inclus,statut),n in b.groupby(['SOURCE','INCLUS_COMPTAGE','STATUT_AFFECTATION'],dropna=False).size().items():
        statistiques[(source,bool(inclus),str(statut))]+=int(n)
    valide=b['COORDONNEES_VALIDES']
    if (valide & (~b['LONGITUDE'].between(-180,180)|~b['LATITUDE'].between(-90,90)|b[['X_2154','Y_2154']].isna().any(axis=1))).any():
        raise ValueError('Point marqué valide avec coordonnées absentes/hors plage.')
    proche=valide & b['X_2154'].between(x0-MARGE_CARTE_METRES,x1+MARGE_CARTE_METRES) & b['Y_2154'].between(y0-MARGE_CARTE_METRES,y1+MARGE_CARTE_METRES)
    b['J13E_AFFECTE_FOCUS']=b['IRIS_AFFECTE'].isin(iris_focus)
    b['J13E_DANS_EMPRISE_CARTE']=proche
    selection=b[b['J13E_AFFECTE_FOCUS']|proche].copy()
    if not selection.empty:locaux.append(selection)
points_focus=pd.concat(locaux,ignore_index=True) if locaux else pd.DataFrame(columns=colonnes_points+['J13E_AFFECTE_FOCUS','J13E_DANS_EMPRISE_CARTE'])
bilan_points=pd.DataFrame([{'SOURCE':s,'INCLUS_COMPTAGE':i,'STATUT_AFFECTATION':st,'NB_POINTS':n}
                          for (s,i,st),n in statistiques.items()],columns=['SOURCE','INCLUS_COMPTAGE','STATUT_AFFECTATION','NB_POINTS'])
for source,col in NOMS_COMPTES.items():
    if not SOURCES_CHARGEES[source]:continue
    attendus=nombre(base[col],col).to_numpy()
    obtenus=np.array([comptes_points[(source,i)] for i in base['IRIS']])
    if not np.array_equal(attendus,obtenus):raise ValueError(source+' : points et comptes IRIS différents ; reprendre J13c.')
rapides=np.array([sum(comptes_categories[("RESTAURATION",i,c)] for c in ['RAPIDE','RESTAURATION_RAPIDE']) for i in base['IRIS']])
if SOURCES_CHARGEES['RESTAURATION'] and not np.array_equal(rapid.to_numpy(),rapides):raise ValueError('Comptes de restauration rapide différents.')
exporter_csv(bilan_points,DOSSIER_INTERIM/'bilan_points_j13e.csv')
print('Points parcourus :',len(cles_vues),' ; points conservés pour le focus/carte :',len(points_focus))
display(bilan_points)


Points parcourus : 554667  ; points conservés pour le focus/carte : 7441


,SOURCE,INCLUS_COMPTAGE,STATUT_AFFECTATION,NB_POINTS
0,RESTAURATION,False,COORDONNEES_ABSENTES_OU_INVALIDES,1313
1,RESTAURATION,False,INTERIEUR_UNIQUE,37
2,RESTAURATION,True,INTERIEUR_UNIQUE,47819
3,TRANSPORTS,False,HORS_IRIS_IDF,410
4,TRANSPORTS,True,INTERIEUR_UNIQUE,37546
5,EQUIPEMENTS,False,COORDONNEES_ABSENTES_OU_INVALIDES,20586
6,EQUIPEMENTS,False,INTERIEUR_UNIQUE,330
7,EQUIPEMENTS,True,INTERIEUR_UNIQUE,446625
8,EQUIPEMENTS,False,HORS_IRIS_IDF,1


## 13 — Catégories en format long et parts internes
La table longue conserve les catégories de restauration, commerces, santé,
services, loisirs, éducation et tourisme déjà préparées dans J13c. Aucun code BPE
n’est réinterprété arbitrairement. Les parts sont recalculées par source :
restaurants parmi les restaurants, équipements parmi les équipements, etc.
Elles ne mélangent pas un arrêt et un établissement dans un total économique.

In [13]:
categories=lire_csv(FICHIER_CATEGORIES)
requis_cat={'IRIS','SOURCE','CATEGORIE','NB_OBSERVATIONS'}
if requis_cat-set(categories):raise ValueError('Catégories J13c : colonnes manquantes.')
if categories[['IRIS','SOURCE','CATEGORIE']].isna().any().any() or categories.duplicated(['IRIS','SOURCE','CATEGORIE']).any():
    raise ValueError('Catégories : clé vide/dupliquée.')
if set(categories['IRIS'])-ATTENDUS or not categories['SOURCE'].isin(NOMS_COMPTES).all():raise ValueError('Catégories hors périmètre/source.')
categories['NB_OBSERVATIONS']=nombre(categories['NB_OBSERVATIONS'],'Catégories — effectif')
if (categories['NB_OBSERVATIONS'].le(0)|(categories['NB_OBSERVATIONS']%1!=0)).any():raise ValueError('Catégorie observée : compte positif entier attendu.')
obtenus={(r.SOURCE,r.IRIS,r.CATEGORIE):int(r.NB_OBSERVATIONS) for r in categories.itertuples()}
if obtenus!=dict(comptes_categories):raise ValueError('Catégories et détail des points incohérents.')
categories_iris=categories[['IRIS','SOURCE','CATEGORIE','NB_OBSERVATIONS']].merge(
    base[['IRIS','J13D_COMMUNE_CONTEXTE']],on='IRIS',validate='many_to_one')
categories_iris['NB_SOURCE_MAILLE']=categories_iris.groupby(['IRIS','SOURCE'])['NB_OBSERVATIONS'].transform('sum')
categories_iris['PART_DANS_SOURCE_PCT']=ratio(categories_iris['NB_OBSERVATIONS'],categories_iris['NB_SOURCE_MAILLE'])
categories_communes=categories_iris.groupby(['J13D_COMMUNE_CONTEXTE','SOURCE','CATEGORIE'],as_index=False)['NB_OBSERVATIONS'].sum()
categories_communes=categories_communes.rename(columns={'J13D_COMMUNE_CONTEXTE':'CODGEO'})
categories_communes['NB_SOURCE_MAILLE']=categories_communes.groupby(['CODGEO','SOURCE'])['NB_OBSERVATIONS'].transform('sum')
categories_communes['PART_DANS_SOURCE_PCT']=ratio(categories_communes['NB_OBSERVATIONS'],categories_communes['NB_SOURCE_MAILLE'])
display(categories_communes[categories_communes['CODGEO'].isin(FOCUS)])


,CODGEO,SOURCE,CATEGORIE,NB_OBSERVATIONS,NB_SOURCE_MAILLE,PART_DANS_SOURCE_PCT
54466,95127,EQUIPEMENTS,TYPE_BPE_A121,3.0,1971.0,0.152207
54467,95127,EQUIPEMENTS,TYPE_BPE_A122,1.0,1971.0,0.050736
54468,95127,EQUIPEMENTS,TYPE_BPE_A124,1.0,1971.0,0.050736
54469,95127,EQUIPEMENTS,TYPE_BPE_A128,3.0,1971.0,0.152207
54470,95127,EQUIPEMENTS,TYPE_BPE_A129,1.0,1971.0,0.050736
...,...,...,...,...,...,...
59429,95500,EQUIPEMENTS,TYPE_BPE_G102,1.0,1202.0,0.083195
59430,95500,EQUIPEMENTS,TYPE_BPE_G104,7.0,1202.0,0.582363
59431,95500,RESTAURATION,RAPIDE,75.0,128.0,58.593750
59432,95500,RESTAURATION,TRADITIONNELLE,53.0,128.0,41.406250


## 14 — Catalogue des indicateurs de diagnostic
Le catalogue choisit un nombre limité d’indicateurs pour l’application et les
comparaisons. Les autres variables restent dans le profil complet. Une variable
absente est signalée ; elle n’est pas remplacée par une variable d’un autre champ.
Les disponibilités sont évaluées indicateur par indicateur.

In [14]:
catalogue=[]
def ajouter(maille,col,libelle,unite,theme,limite,position=False):
    catalogue.append({'MAILLE':maille,'COLONNE':col,'LIBELLE':libelle,'UNITE':unite,
        'THEME':theme,'LIMITE':limite,'POSITION_RELATIVE':position})
for maille in ['IRIS','COMMUNE']:
    ajouter(maille,'J13E_POPULATION','Population résidente','habitants','POPULATION','Millésime RP ; aucune présence diurne estimée.')
    ajouter(maille,'J13E_PART_1529_PCT','Part des 15–29 ans','%','POPULATION','Ne décrit pas uniquement des étudiants.',True)
    ajouter(maille,'J13E_NB_OBS_RESTAURATION','Observations de restauration retenues','observations','OFFRE','Couverture de la source et des affectations à contrôler.')
    ajouter(maille,'J13E_NB_OBS_RESTAURATION_RAPIDE','Observations de restauration rapide retenues','observations','OFFRE','Catégories documentées de J13c.')
    ajouter(maille,'J13E_DENSITE_OBS_RESTAURATION_10000_HAB','Restauration pour 10 000 habitants','observations / 10 000 hab.','OFFRE','Rapport à la population résidente ; pas un seuil de saturation.',True)
    ajouter(maille,'J13E_DENSITE_OBS_RAPIDE_10000_HAB','Restauration rapide pour 10 000 habitants','observations / 10 000 hab.','OFFRE','Sensibilité aux petits dénominateurs.',True)
    ajouter(maille,'J13E_PART_RESTAURATION_RAPIDE_PCT','Part de restauration rapide dans l’offre retenue','%','OFFRE','NA si aucun restaurant retenu.')
    ajouter(maille,'J13E_NB_POINTS_ARRET','Points de transport retenus','points selon unité J13c','TRANSPORTS','Pas nécessairement gares uniques ; ne mesure pas la desserte.')
    ajouter(maille,'J13E_DENSITE_POINTS_ARRET_KM2','Points de transport par km²','points / km²','TRANSPORTS','Localisation selon grain de la source ; pas fréquence/accessibilité.',True)
    ajouter(maille,'J13E_NB_OBS_EQUIPEMENTS','Équipements complémentaires retenus','observations','EQUIPEMENTS','Hétérogènes ; détails disponibles dans les catégories.')
for col,lib,un,theme in [
    (COL_MED,'Niveau de vie médian annuel par unité de consommation','euros / UC','REVENUS'),
    (COL_PAUV,'Taux de pauvreté au seuil de 60 %','%','REVENUS'),
    ('J13B_PART_MENAGES_UNE_PERSONNE_PCT','Part des ménages d’une personne','%','MENAGES'),
    ('J13B_PART_CADRES_PARMI_ACTIFS_OCCUPES_15_64_PCT','Part des cadres parmi les actifs occupés de 15–64 ans','%','ACTIVITE'),
    ('J13B_PART_DIPLOMES_SUPERIEUR_NON_SCOLARISES_15P_PCT','Part de diplômés du supérieur parmi les non-scolarisés de 15 ans ou plus','%','FORMATION')]:
    ajouter('IRIS',col,lib,un,theme,'Valeur et statut hérités J13b ; non agrégés en fiscal communal.',col==COL_MED)
for col,lib,un in [
    ('J13D_MOBPRO_LIEU','Actifs occupés déclarant travailler dans la commune','personnes pondérées'),
    ('J13D_MOBPRO_PART_ENTRANTS_LIEU_PCT','Part des entrants parmi les personnes au lieu de travail','%'),
    ('J13D_MOBPRO_RATIO_LIEU_RESIDENTS','Ratio effectif au lieu de travail / effectif résident du même champ','ratio'),
    ('J13D_MOBSCO_LIEU','Personnes scolarisées déclarant étudier dans la commune','personnes pondérées'),
    ('J13D_MOBSCO_1824_LIEU_ETUDE','Scolarisés de 18–24 ans au lieu d’études','personnes pondérées'),
    ('J13E_OBS_RESTAURATION_1000_PERSONNES_LIEU_TRAVAIL','Restauration pour 1 000 personnes au lieu de travail','observations / 1 000 personnes')]:
    ajouter('COMMUNE',col,lib,un,'MOBILITES','Maille commune uniquement ; pas présence observée à midi.',col=='J13D_MOBPRO_LIEU')
catalogue=pd.DataFrame(catalogue)
if catalogue.duplicated(['MAILLE','COLONNE']).any():raise ValueError('Catalogue dupliqué.')
display(catalogue[['MAILLE','LIBELLE','UNITE','POSITION_RELATIVE']])


,MAILLE,LIBELLE,UNITE,POSITION_RELATIVE
0,IRIS,Population résidente,habitants,False
1,IRIS,Part des 15–29 ans,%,True
2,IRIS,Observations de restauration retenues,observations,False
3,IRIS,Observations de restauration rapide retenues,observations,False
4,IRIS,Restauration pour 10 000 habitants,observations / 10 000 hab.,True
...,...,...,...,...
26,COMMUNE,Part des entrants parmi les personnes au lieu ...,%,False
27,COMMUNE,Ratio effectif au lieu de travail / effectif r...,ratio,False
28,COMMUNE,Personnes scolarisées déclarant étudier dans l...,personnes pondérées,False
29,COMMUNE,Scolarisés de 18–24 ans au lieu d’études,personnes pondérées,False


## 15 — Couverture des indicateurs et éligibilité aux comparaisons
Les profils restent complets. Les seuils de population filtrent seulement la
référence statistique et les tableaux de comparaison. Les IRIS d’activité ne sont
pas écartés du diagnostic local : ils peuvent simplement ne pas entrer dans une
comparaison par habitant lorsqu’ils sont peu peuplés.

In [15]:
profil_iris['J13E_ELIGIBLE_COMPARAISON']=profil_iris['J13E_POPULATION'].ge(MIN_POP_IRIS_COMPARAISON)
communes['J13E_ELIGIBLE_COMPARAISON']=communes['J13E_POPULATION'].ge(MIN_POP_COMMUNE_COMPARAISON)
couverture=[]
for maille,t in [('IRIS',profil_iris),('COMMUNE',communes)]:
    for r in catalogue[catalogue['MAILLE'].eq(maille)].itertuples():
        s=serie_numerique(t,r.COLONNE)
        couverture.append({'MAILLE':maille,'COLONNE':r.COLONNE,'LIBELLE':r.LIBELLE,
            'COLONNE_PRESENTE':r.COLONNE in t,'NB_MAILLES':len(t),'NB_DISPONIBLES':int(s.notna().sum()),
            'COUVERTURE_PCT':100*s.notna().mean(),'NB_ELIGIBLES_RENSEIGNES':int((t['J13E_ELIGIBLE_COMPARAISON']&s.notna()).sum())})
couverture=pd.DataFrame(couverture)
exporter_csv(couverture,DOSSIER_INTERIM/'couverture_indicateurs_j13e.csv')
display(couverture)


,MAILLE,COLONNE,LIBELLE,COLONNE_PRESENTE,NB_MAILLES,NB_DISPONIBLES,COUVERTURE_PCT,NB_ELIGIBLES_RENSEIGNES
0,IRIS,J13E_POPULATION,Population résidente,True,5264,5264,100.000000,4801
1,IRIS,J13E_PART_1529_PCT,Part des 15–29 ans,True,5264,5208,98.936170,4801
2,IRIS,J13E_NB_OBS_RESTAURATION,Observations de restauration retenues,True,5264,5264,100.000000,4801
3,IRIS,J13E_NB_OBS_RESTAURATION_RAPIDE,Observations de restauration rapide retenues,True,5264,5264,100.000000,4801
4,IRIS,J13E_DENSITE_OBS_RESTAURATION_10000_HAB,Restauration pour 10 000 habitants,True,5264,5208,98.936170,4801
...,...,...,...,...,...,...,...,...
26,COMMUNE,J13D_MOBPRO_PART_ENTRANTS_LIEU_PCT,Part des entrants parmi les personnes au lieu ...,True,1267,1265,99.842147,376
27,COMMUNE,J13D_MOBPRO_RATIO_LIEU_RESIDENTS,Ratio effectif au lieu de travail / effectif r...,True,1267,1267,100.000000,376
28,COMMUNE,J13D_MOBSCO_LIEU,Personnes scolarisées déclarant étudier dans l...,True,1267,1267,100.000000,376
29,COMMUNE,J13D_MOBSCO_1824_LIEU_ETUDE,Scolarisés de 18–24 ans au lieu d’études,True,1267,1267,100.000000,376


## 16 — Positions relatives transparentes, sans score global
Pour chaque indicateur sélectionné : position = 100 × (rang moyen − 1)/(n − 1).
Les valeurs égales ont la même position. Si toutes les valeurs sont égales, leur
position est 50. Si la référence compte moins de MIN_REFERENCE valeurs, la position
reste NA. Les IRIS sont comparés à leur type dans toute l’IDF ; les communes à
l’ensemble des communes éligibles d’IDF. Une position élevée signifie une valeur
élevée, pas une meilleure implantation. Aucun indicateur n’est inversé ou pondéré.

In [16]:
references=[]
def calculer_positions(t,maille):
    for r in catalogue[catalogue['MAILLE'].eq(maille)&catalogue['POSITION_RELATIVE']].itertuples():
        col_pos='J13E_POSITION_'+r.COLONNE
        t[col_pos]=np.nan
        groupes=t.groupby('J13E_TYPE_IRIS_COMPARAISON').groups if maille=='IRIS' else {'COMMUNES_IDF':t.index}
        valeurs=serie_numerique(t,r.COLONNE)
        for groupe,ix in groupes.items():
            masque=t.loc[ix,'J13E_ELIGIBLE_COMPARAISON'] & valeurs.loc[ix].notna()
            selection=valeurs.loc[ix][masque]
            n=len(selection)
            q25=selection.quantile(.25) if n else np.nan
            q75=selection.quantile(.75) if n else np.nan
            statut='REFERENCE_INSUFFISANTE' if n<MIN_REFERENCE else ('TOUTES_VALEURS_IDENTIQUES' if selection.nunique()==1 else 'COMPARABLE')
            if n>=MIN_REFERENCE:t.loc[selection.index,col_pos]=100*(selection.rank(method='average')-1)/(n-1)
            references.append({'MAILLE':maille,'GROUPE_REFERENCE':str(groupe),'COLONNE':r.COLONNE,
                'LIBELLE':r.LIBELLE,'UNITE':r.UNITE,'N_REFERENCE':n,'Q25':q25,'MEDIANE':selection.median() if n else np.nan,
                'Q75':q75,'STATUT':statut,'FORMULE_POSITION':'100 * (rang moyen - 1) / (n - 1)'})
    return t
profil_iris=calculer_positions(profil_iris,'IRIS')
communes=calculer_positions(communes,'COMMUNE')
references=pd.DataFrame(references)
exporter_csv(references,DOSSIER_PROCESSED/'references_comparaison_j13e.csv')
display(references)


,MAILLE,GROUPE_REFERENCE,COLONNE,LIBELLE,UNITE,N_REFERENCE,Q25,MEDIANE,Q75,STATUT,FORMULE_POSITION
0,IRIS,A,J13E_PART_1529_PCT,Part des 15–29 ans,%,67,19.737559,25.147845,29.486525,COMPARABLE,100 * (rang moyen - 1) / (n - 1)
1,IRIS,D,J13E_PART_1529_PCT,Part des 15–29 ans,%,18,18.703643,22.268524,24.579766,COMPARABLE,100 * (rang moyen - 1) / (n - 1)
2,IRIS,H,J13E_PART_1529_PCT,Part des 15–29 ans,%,4065,17.422694,19.725837,22.709183,COMPARABLE,100 * (rang moyen - 1) / (n - 1)
3,IRIS,Z,J13E_PART_1529_PCT,Part des 15–29 ans,%,651,14.163914,15.592001,17.227441,COMPARABLE,100 * (rang moyen - 1) / (n - 1)
4,IRIS,A,J13E_DENSITE_OBS_RESTAURATION_10000_HAB,Restauration pour 10 000 habitants,observations / 10 000 hab.,67,49.955933,146.222796,355.266207,COMPARABLE,100 * (rang moyen - 1) / (n - 1)
5,IRIS,D,J13E_DENSITE_OBS_RESTAURATION_10000_HAB,Restauration pour 10 000 habitants,observations / 10 000 hab.,18,6.222950,18.679264,40.255433,COMPARABLE,100 * (rang moyen - 1) / (n - 1)
6,IRIS,H,J13E_DENSITE_OBS_RESTAURATION_10000_HAB,Restauration pour 10 000 habitants,observations / 10 000 hab.,4065,10.546628,23.261524,46.646100,COMPARABLE,100 * (rang moyen - 1) / (n - 1)
7,IRIS,Z,J13E_DENSITE_OBS_RESTAURATION_10000_HAB,Restauration pour 10 000 habitants,observations / 10 000 hab.,651,3.603020,13.568521,22.618850,COMPARABLE,100 * (rang moyen - 1) / (n - 1)
8,IRIS,A,J13E_DENSITE_OBS_RAPIDE_10000_HAB,Restauration rapide pour 10 000 habitants,observations / 10 000 hab.,67,27.675994,74.488274,200.042901,COMPARABLE,100 * (rang moyen - 1) / (n - 1)
9,IRIS,D,J13E_DENSITE_OBS_RAPIDE_10000_HAB,Restauration rapide pour 10 000 habitants,observations / 10 000 hab.,18,0.000000,9.595317,16.423164,COMPARABLE,100 * (rang moyen - 1) / (n - 1)


## 17 — Table de comparaison de Cergy-Pontoise
Le format long sépare libellé, valeur, unité, maille, disponibilité et groupe de
référence. Les quartiles sont descriptifs. Une valeur hors référence n’est pas
qualifiée faible ou forte. Les limites accompagnent chaque observation.

In [17]:
diagnostic=[]
def millesimes(ligne,theme,maille):
    if theme=='REVENUS':
        return ligne.get('J13B_ANNEE_FILOSOFI_SOURCE',pd.NA),ligne.get('J13B_GEOGRAPHIE_FILOSOFI_SOURCE',pd.NA)
    source={'OFFRE':'RESTAURATION','TRANSPORTS':'TRANSPORTS','EQUIPEMENTS':'EQUIPEMENTS'}.get(theme)
    if source:
        p=meta_c.get('provenance_sources_declarees',{}).get(source,{})
        return p.get('annee_source'),p.get('geographie_source')
    return ANNEE_RP,GEOGRAPHIE
for maille,t,cle_t,nom in [('IRIS',profil_iris,'IRIS','NOM_IRIS'),('COMMUNE',communes,'CODGEO','NOM_COMMUNE')]:
    for r in catalogue[catalogue['MAILLE'].eq(maille)].itertuples():
        valeurs=serie_numerique(t,r.COLONNE)
        for ix,ligne in t[t['J13E_DANS_FOCUS']].iterrows():
            valeur=valeurs.loc[ix]
            annee_source,geographie_source=millesimes(ligne,r.THEME,maille)
            groupe=str(ligne['J13E_TYPE_IRIS_COMPARAISON']) if maille=='IRIS' else 'COMMUNES_IDF'
            ref=references[(references['MAILLE']==maille)&(references['GROUPE_REFERENCE']==groupe)&(references['COLONNE']==r.COLONNE)]
            lecture='INDICATEUR_DESCRIPTIF'
            position=ligne.get('J13E_POSITION_'+r.COLONNE,np.nan)
            n=int(ref.iloc[0]['N_REFERENCE']) if not ref.empty else 0
            if pd.isna(valeur):lecture='DONNEE_NON_DISPONIBLE'
            elif r.POSITION_RELATIVE:
                if not ligne['J13E_ELIGIBLE_COMPARAISON']:lecture='HORS_REFERENCE_SEUIL_POPULATION'
                elif n<MIN_REFERENCE:lecture='REFERENCE_INSUFFISANTE'
                elif ref.iloc[0]['STATUT']=='TOUTES_VALEURS_IDENTIQUES':lecture='MEME_VALEUR_DANS_LA_REFERENCE'
                elif valeur<ref.iloc[0]['Q25']:lecture='SOUS_LE_PREMIER_QUARTILE'
                elif valeur>ref.iloc[0]['Q75']:lecture='AU_DESSUS_DU_TROISIEME_QUARTILE'
                else:lecture='ENTRE_LES_QUARTILES'
            diagnostic.append({'MAILLE':maille,'CODE':ligne[cle_t],'NOM':ligne[nom],
                'CODGEO_CONTEXTE':ligne['J13D_COMMUNE_CONTEXTE'] if maille=='IRIS' else ligne['CODGEO'],
                'THEME':r.THEME,'COLONNE':r.COLONNE,'LIBELLE':r.LIBELLE,'VALEUR':valeur,'UNITE':r.UNITE,
                'DISPONIBLE':pd.notna(valeur),'GROUPE_REFERENCE':groupe,'N_REFERENCE':n,
                'POSITION_RELATIVE_0_100':position,'LECTURE':lecture,'LIMITE':r.LIMITE,
                'ANNEE_DONNEES':annee_source,'ANNEE_GEOGRAPHIE_SOURCE':geographie_source,'GEOGRAPHIE_ANALYSE':GEOGRAPHIE})
diagnostic=pd.DataFrame(diagnostic)
exporter_csv(diagnostic,DOSSIER_PROCESSED/'diagnostic_focus_long_j13e.csv')
display(diagnostic.head(25))


,MAILLE,CODE,NOM,CODGEO_CONTEXTE,THEME,COLONNE,LIBELLE,VALEUR,UNITE,DISPONIBLE,GROUPE_REFERENCE,N_REFERENCE,POSITION_RELATIVE_0_100,LECTURE,LIMITE,ANNEE_DONNEES,ANNEE_GEOGRAPHIE_SOURCE,GEOGRAPHIE_ANALYSE
0,IRIS,955000304,Saint-Martin,95500,POPULATION,J13E_POPULATION,Population résidente,4262.914601,habitants,True,H,0,NaN,INDICATEUR_DESCRIPTIF,Millésime RP ; aucune présence diurne estimée.,2022,2024,2024
1,IRIS,955000103,Hermitage,95500,POPULATION,J13E_POPULATION,Population résidente,3090.766322,habitants,True,H,0,NaN,INDICATEUR_DESCRIPTIF,Millésime RP ; aucune présence diurne estimée.,2022,2024,2024
2,IRIS,951270106,Port Ham-Village,95127,POPULATION,J13E_POPULATION,Population résidente,3856.871645,habitants,True,H,0,NaN,INDICATEUR_DESCRIPTIF,Millésime RP ; aucune présence diurne estimée.,2022,2024,2024
3,IRIS,951270113,La Sébille,95127,POPULATION,J13E_POPULATION,Population résidente,2244.011600,habitants,True,H,0,NaN,INDICATEUR_DESCRIPTIF,Millésime RP ; aucune présence diurne estimée.,2022,2024,2024
4,IRIS,951270127,Le Bontemps,95127,POPULATION,J13E_POPULATION,Population résidente,2358.520974,habitants,True,H,0,NaN,INDICATEUR_DESCRIPTIF,Millésime RP ; aucune présence diurne estimée.,2022,2024,2024
5,IRIS,951270108,Zone Agricole,95127,POPULATION,J13E_POPULATION,Population résidente,22.850045,habitants,True,D,0,NaN,INDICATEUR_DESCRIPTIF,Millésime RP ; aucune présence diurne estimée.,2022,2024,2024
6,IRIS,951270122,Le Gros Caillou,95127,POPULATION,J13E_POPULATION,Population résidente,1361.217954,habitants,True,H,0,NaN,INDICATEUR_DESCRIPTIF,Millésime RP ; aucune présence diurne estimée.,2022,2024,2024
7,IRIS,955000201,Centre Ville,95500,POPULATION,J13E_POPULATION,Population résidente,4281.310497,habitants,True,H,0,NaN,INDICATEUR_DESCRIPTIF,Millésime RP ; aucune présence diurne estimée.,2022,2024,2024
8,IRIS,951270109,Cité Artisanale-Chauffours,95127,POPULATION,J13E_POPULATION,Population résidente,549.311089,habitants,True,A,0,NaN,INDICATEUR_DESCRIPTIF,Millésime RP ; aucune présence diurne estimée.,2022,2024,2024
9,IRIS,955000301,Saint-Martin-Zone d'Activités,95500,POPULATION,J13E_POPULATION,Population résidente,1108.943699,habitants,True,A,0,NaN,INDICATEUR_DESCRIPTIF,Millésime RP ; aucune présence diurne estimée.,2022,2024,2024


## 18 — Questions marketing et vérifications terrain
Ces questions structurent une étude client sans produire une recommandation
d’investissement automatique. Le tableau est un plan d’investigation. Les loyers,
surfaces, prix, horaires et passages ne sont pas inventés depuis la BPE.

In [18]:
questions=pd.DataFrame([
 {'SUJET':'Restauration du midi','DONNEES':'MOBPRO au lieu de travail, restauration J13c',
  'QUESTION':'Quels secteurs accueillent des actifs et quelle offre de repas existe réellement ?',
  'A_VERIFIER':'Localisation des emplois, télétravail, horaires, prix, passages à midi'},
 {'SUJET':'Jeunes et scolarisés','DONNEES':'Âges résidents IRIS ; MOBSCO 18–24 au lieu d’études communal',
  'QUESTION':'Quels besoins de restauration peuvent être étudiés autour des sites d’enseignement ?',
  'A_VERIFIER':'Campus, niveau d’études, effectifs présents, CROUS, budget et calendrier'},
 {'SUJET':'Offre de proximité','DONNEES':'Population, ménages, revenus IRIS et catégories BPE',
  'QUESTION':'Comment l’offre locale se répartit-elle selon les profils résidentiels ?',
  'A_VERIFIER':'Commerces voisins hors IRIS, pratiques d’achat, concurrence, prix'},
 {'SUJET':'Transports','DONNEES':'Points d’arrêt selon la source IDFM utilisée',
  'QUESTION':'Quels lieux et cheminements relient les arrêts aux établissements ?',
  'A_VERIFIER':'Fréquence, fréquentation, correspondances, barrières et marche réelle'}])
exporter_csv(questions,DOSSIER_PROCESSED/'questions_terrain_j13e.csv')
display(questions)


,SUJET,DONNEES,QUESTION,A_VERIFIER
0,Restauration du midi,"MOBPRO au lieu de travail, restauration J13c",Quels secteurs accueillent des actifs et quell...,"Localisation des emplois, télétravail, horaire..."
1,Jeunes et scolarisés,Âges résidents IRIS ; MOBSCO 18–24 au lieu d’é...,Quels besoins de restauration peuvent être étu...,"Campus, niveau d’études, effectifs présents, C..."
2,Offre de proximité,"Population, ménages, revenus IRIS et catégorie...",Comment l’offre locale se répartit-elle selon ...,"Commerces voisins hors IRIS, pratiques d’achat..."
3,Transports,Points d’arrêt selon la source IDFM utilisée,Quels lieux et cheminements relient les arrêts...,"Fréquence, fréquentation, correspondances, bar..."


## 19 — Exports complets, focus et GeoPackages
Les CSV complets conservent les colonnes antérieures ; les tables focus sont des
extraits supplémentaires. Aucun ancien fichier n’est écrasé. Le GPKG contient
les contours complets, destinés aux analyses futures plutôt qu’au rendu web.

In [19]:
iris_focus_table=profil_iris[profil_iris['J13E_DANS_FOCUS']].copy()
communes_focus=communes[communes['J13E_DANS_FOCUS']].copy()
for nom,t in [
 ('profil_iris_idf_j13e.csv',profil_iris),('profil_communes_idf_j13e.csv',communes),
 ('profil_iris_focus_j13e.csv',iris_focus_table),('profil_communes_focus_j13e.csv',communes_focus),
 ('points_focus_j13e.csv',points_focus),('categories_iris_j13e.csv',categories_iris),
 ('categories_communes_j13e.csv',categories_communes)]:exporter_csv(t,DOSSIER_PROCESSED/nom)
for geo,t,cle_t,nom,couche in [
 (iris_geo,profil_iris,'IRIS','profil_iris_idf_j13e.gpkg','profils_iris'),
 (geo_communes,communes,'CODGEO','profil_communes_idf_j13e.gpkg','profils_communes')]:
    sortie=DOSSIER_PROCESSED/nom;temp=sortie.with_name(sortie.stem+'_temp.gpkg')
    if temp.exists():temp.unlink()
    geo.merge(t,on=cle_t,validate='one_to_one').to_file(temp,layer=couche,driver='GPKG',index=False)
    temp.replace(sortie);SORTIES.append(sortie)
print('Profils complets exportés ; IRIS :',len(profil_iris),' ; communes :',len(communes))


Profils complets exportés ; IRIS : 5264  ; communes : 1267


## 20 — Cartes Plotly sans clé API ni fond OpenStreetMap
Les cartes locales montrent population, offre de restauration et revenus
disponibles. Le gris signifie donnée absente. Les valeurs manquantes restent
explicites au survol ; les noms ne sont jamais affichés en permanence sur la carte.
Les contours simplifiés servent seulement à l’affichage. Les HTML sont écrits
séparément, pour éviter d’alourdir le notebook.

In [20]:
FICHIERS_HTML=[]
def fmt(v,dec=1):return 'Non disponible' if pd.isna(v) else f'{float(v):,.{dec}f}'.replace(',',' ').replace('.',',')
def sauver_figure(fig,nom,dossier=DOSSIER_CARTES):
    p=dossier/nom
    fig.write_html(str(p),include_plotlyjs=True,full_html=True,config=CONFIG_PLOTLY)
    if p not in FICHIERS_HTML:FICHIERS_HTML.append(p)
    if p not in SORTIES:SORTIES.append(p)
    if AFFICHER_CARTES:fig.show(config=CONFIG_PLOTLY)
    return p
geo_affichage=geo_focus.copy()
geo_affichage.geometry=geo_affichage.geometry.simplify(SIMPLIFICATION_CARTE_METRES,preserve_topology=True)
geo_affichage=geo_affichage.to_crs(4326).merge(iris_focus_table,on='IRIS',validate='one_to_one')
geojson_focus=json.loads(geo_affichage[['IRIS','geometry']].to_json())
def carte_indicateur(col,libelle,unite,nom):
    valeur=serie_numerique(geo_affichage,col)
    fig=go.Figure(go.Choropleth(geojson=geojson_focus,featureidkey='properties.IRIS',locations=geo_affichage['IRIS'],
        z=np.zeros(len(geo_affichage)),colorscale=[[0,'#dfe4ea'],[1,'#dfe4ea']],showscale=False,
        text=geo_affichage['NOM_IRIS'],hovertemplate='%{text}<br>Donnée non disponible<extra></extra>',
        marker_line_color='#ffffff',marker_line_width=.5))
    ok=valeur.notna()
    if ok.any():
        survol=[html.escape(str(r['NOM_IRIS']))+'<br>IRIS : '+str(r['IRIS'])+'<br>'+html.escape(libelle)+
                ' : '+fmt(valeur.loc[ix])+' '+html.escape(unite) for ix,r in geo_affichage.loc[ok].iterrows()]
        fig.add_trace(go.Choropleth(geojson=geojson_focus,featureidkey='properties.IRIS',locations=geo_affichage.loc[ok,'IRIS'],
            z=valeur[ok],colorscale='Viridis',colorbar_title=unite,text=survol,
            hovertemplate='%{text}<extra></extra>',marker_line_color='#ffffff',marker_line_width=.5))
    fig.update_geos(visible=False,fitbounds='locations',projection_type='mercator')
    fig.update_layout(title=libelle+' — IRIS du focus '+LIBELLE_FOCUS,height=700,margin=dict(l=10,r=10,t=65,b=10))
    return sauver_figure(fig,nom)
carte_indicateur('J13E_POPULATION','Population résidente','habitants','01_population_iris_focus.html')
carte_indicateur('J13E_DENSITE_OBS_RESTAURATION_10000_HAB','Restauration retenue pour 10 000 habitants','observations / 10 000 hab.','02_restauration_iris_focus.html')
carte_indicateur(COL_MED,'Niveau de vie médian annuel','euros / UC','03_revenus_iris_focus.html')


WindowsPath('C:/Users/almou/OneDrive/GeoMarketing_IDF/outputs/cartes/j13e/03_revenus_iris_focus.html')

## 21 — Restaurants, arrêts et équipements sur la carte locale
Les points valides inclus et exclus du comptage sont distingués. Les points
proches du focus peuvent appartenir à un autre IRIS ; le survol le précise.
La sélection dans une emprise rectangulaire n’est pas un calcul de voisinage.
Une catégorie ou un nom absent n’est pas inventé. Le statut explique pourquoi
un point ne contribue pas aux comptes du profil.

In [21]:
p=points_focus[points_focus['COORDONNEES_VALIDES'].eq(True)&points_focus['J13E_DANS_EMPRISE_CARTE'].eq(True)].copy()
if len(p)>MAX_POINTS_CARTE:raise ValueError('Trop de points pour cette carte ; réduire le focus ou augmenter MAX_POINTS_CARTE explicitement.')
fig=go.Figure(go.Choropleth(geojson=geojson_focus,featureidkey='properties.IRIS',locations=geo_affichage['IRIS'],
    z=np.zeros(len(geo_affichage)),colorscale=[[0,'#f0f3f7'],[1,'#f0f3f7']],showscale=False,
    text=geo_affichage['NOM_IRIS'],hovertemplate='%{text}<extra></extra>',marker_line_color='#8b95a5',marker_line_width=.7))
couleurs={'RESTAURATION':'#e76f51','TRANSPORTS':'#277da8','EQUIPEMENTS':'#43aa8b'}
for (source,inclus),g in p.groupby(['SOURCE','INCLUS_COMPTAGE']):
    survol=[]
    for r in g.itertuples():
        nom='Nom absent de la source' if pd.isna(r.NOM) else str(r.NOM)
        iris='Non affecté' if pd.isna(r.IRIS_AFFECTE) else str(r.IRIS_AFFECTE)
        survol.append(html.escape(nom)+'<br>Source : '+source+'<br>Catégorie : '+html.escape(str(r.CATEGORIE))+
            '<br>IRIS : '+iris+'<br>Compté : '+('Oui' if inclus else 'Non')+'<br>Statut : '+html.escape(str(r.STATUT_AFFECTATION)))
    fig.add_trace(go.Scattergeo(lon=g['LONGITUDE'],lat=g['LATITUDE'],mode='markers',
        name=source+(' — comptés' if inclus else ' — à contrôler'),text=survol,hovertemplate='%{text}<extra></extra>',
        marker=dict(size=7,color=couleurs[source],opacity=.85,symbol='circle' if inclus else 'x')))
fig.update_geos(visible=False,fitbounds='locations',projection_type='mercator')
fig.update_layout(title='Offre géolocalisée et points de transport — focus',height=750,
                  margin=dict(l=10,r=10,t=65,b=10),legend=dict(orientation='h'))
sauver_figure(fig,'04_points_et_iris_focus.html')


WindowsPath('C:/Users/almou/OneDrive/GeoMarketing_IDF/outputs/cartes/j13e/04_points_et_iris_focus.html')

## 22 — Graphiques et comparaison lisible
Les effectifs et densités restent séparés. Les champs travail et études ne sont
pas additionnés. Une figure vide porte explicitement la mention donnée indisponible.
Chaque graphique indique sa maille et son unité, pour éviter une lecture ambiguë.

In [22]:
def barres_communes(col,libelle,unite,nom):
    t=communes_focus.copy();s=serie_numerique(t,col)
    ok=s.notna();fig=go.Figure()
    if ok.any():fig.add_trace(go.Bar(x=t.loc[ok,'NOM_COMMUNE'],y=s[ok],text=[fmt(x) for x in s[ok]],textposition='outside'))
    else:fig.add_annotation(text='Donnée non disponible pour le focus',showarrow=False)
    fig.update_layout(title=libelle+' — communes',yaxis_title=unite,height=450,template='plotly_white')
    sauver_figure(fig,nom)
barres_communes('J13E_NB_OBS_RESTAURATION','Observations de restauration retenues','observations','05_comparaison_offre.html')
barres_communes('J13E_DENSITE_OBS_RESTAURATION_10000_HAB','Restauration pour 10 000 habitants','observations / 10 000 hab.','06_comparaison_densite.html')
barres_communes('J13D_MOBPRO_LIEU','Actifs occupés au lieu de travail déclaré','personnes pondérées','07_comparaison_travail.html')
barres_communes('J13D_MOBSCO_1824_LIEU_ETUDE','Scolarisés de 18–24 ans au lieu d’études','personnes pondérées','08_comparaison_scolarises_1824.html')


## 23 — Fiches locales et rapport HTML client
Le rapport réunit les chiffres et les questions à approfondir, sans déclarer un
secteur rentable. Les revenus apparaissent dans les fiches IRIS et leur couverture
dans les fiches communales. Les cartes restent dans un dossier séparé. Le rapport
est un livrable de démonstration après validation des données locales.

In [23]:
fiches=[]
for r in diagnostic.itertuples():
    fiches.append({'MAILLE':r.MAILLE,'CODE':r.CODE,'NOM':r.NOM,'THEME':r.THEME,
        'OBSERVATION':r.LIBELLE+' : '+fmt(r.VALEUR)+' '+r.UNITE,
        'ANNEE_DONNEES':r.ANNEE_DONNEES,'GEOGRAPHIE_SOURCE':r.ANNEE_GEOGRAPHIE_SOURCE,
        'LECTURE':r.LECTURE,'A_PRECISER':r.LIMITE})
fiches=pd.DataFrame(fiches)
exporter_csv(fiches,DOSSIER_PROCESSED/'fiches_focus_j13e.csv')
styles='body{font-family:Arial,sans-serif;max-width:1200px;margin:32px auto;padding:0 20px;color:#233044}table{border-collapse:collapse;width:100%;font-size:14px}th,td{border:1px solid #d9e0e9;padding:8px;text-align:left}th{background:#edf2f7}.note{background:#fff5d6;padding:14px}h2{margin-top:32px}'
parties=['<!doctype html><html lang="fr"><meta charset="utf-8"><title>Diagnostic territorial J13e</title><style>'+styles+'</style><body>',
 '<h1>Diagnostic de marché — '+html.escape(LIBELLE_FOCUS)+'</h1>',
 '<p>RP '+str(ANNEE_RP)+' ; géographie '+str(GEOGRAPHIE)+'. Les dates BPE/IDFM peuvent différer.</p>',
 '<p class="note">Analyse descriptive. Les mobilités sont communales ; les revenus sont des indicateurs IRIS. Aucune fréquentation ou rentabilité n’est estimée.</p>']
for (maille,code_t,nom),g in fiches.groupby(['MAILLE','CODE','NOM'],sort=True):
    parties.append('<h2>'+html.escape(maille+' '+str(nom)+' ('+str(code_t)+')')+'</h2>')
    parties.append(g[['THEME','OBSERVATION','ANNEE_DONNEES','GEOGRAPHIE_SOURCE','LECTURE','A_PRECISER']].to_html(index=False,escape=True))
parties.extend(['<h2>Couverture fiscale par commune</h2>',couverture_revenus[couverture_revenus['CODGEO'].isin(FOCUS)].to_html(index=False,escape=True),
 '<h2>Questions à vérifier sur le terrain</h2>',questions.to_html(index=False,escape=True),
 '<h2>Disponibilité des indicateurs</h2>',couverture.to_html(index=False,escape=True),'</body></html>'])
FICHIER_RAPPORT=DOSSIER_RAPPORTS/'diagnostic_marche_focus_j13e.html'
FICHIER_RAPPORT.write_text('\n'.join(parties),encoding='utf-8');SORTIES.append(FICHIER_RAPPORT)
print('Rapport :',FICHIER_RAPPORT)


Rapport : C:\Users\almou\OneDrive\GeoMarketing_IDF\outputs\rapports\j13e\diagnostic_marche_focus_j13e.html


## 24 — Tables légères pour l’application
J14 lira ces exports sans recharger les fichiers nationaux. Le profil complet
reste dans processed ; l’application dispose d’un catalogue, de valeurs choisies,
des statuts et de contours simplifiés. Les points sont limités au focus et à sa
marge, annoncée dans le manifeste. Les géométries IDF restent toutes présentes.
Un champ manquant est conservé vide dans l’export applicatif et signalé dans la
couverture. Aucun flux communal n’est ajouté comme mesure locale à la table IRIS.

In [24]:
def table_app(t,maille,identifiants,statuts):
    cols=list(dict.fromkeys(identifiants+catalogue.loc[catalogue['MAILLE'].eq(maille),'COLONNE'].tolist()+
             [c for c in t if c.startswith('J13E_POSITION_')]+statuts))
    out=pd.DataFrame(index=t.index)
    numeriques=set(catalogue.loc[catalogue['MAILLE'].eq(maille),'COLONNE'])
    for col in cols:
        out[col]=serie_numerique(t,col) if col in numeriques else (t[col] if col in t else pd.NA)
    return out
statuts_iris=[c for c in base if c.startswith('J13B_') and any(x in c for x in ['STATUT','DISPONIBLE','FILOSOFI_SOURCE'])]
statuts_iris+=['J13C_SOURCE_'+s+'_CHARGEE' for s in SOURCES_CHARGEES]
app_iris=table_app(profil_iris,'IRIS',['IRIS','NOM_IRIS','CODGEO_SOURCE','CODGEO_ETUDE','J13D_COMMUNE_CONTEXTE',
    'J13E_NOM_COMMUNE_CONTEXTE','J13E_TYPE_IRIS_COMPARAISON','J13E_DANS_FOCUS','J13E_ELIGIBLE_COMPARAISON'],statuts_iris)
app_communes=table_app(communes,'COMMUNE',['CODGEO','NOM_COMMUNE','J13E_DANS_FOCUS','J13E_ELIGIBLE_COMPARAISON'],
    ['J13D_MOBPRO_CHARGE','J13D_MOBSCO_CHARGE','J13D_MOBSCO_AGES_DISPONIBLES','J13E_NB_IRIS',
     'J13E_COUVERTURE_POPULATION_IRIS_PCT']+['J13E_SOURCE_'+s+'_CHARGEE' for s in SOURCES_CHARGEES])
exporter_csv(app_iris,DOSSIER_APP/'iris_idf.csv')
exporter_csv(app_communes,DOSSIER_APP/'communes_idf.csv')
exporter_csv(points_focus,DOSSIER_APP/'points_focus.csv')
exporter_csv(categories_iris,DOSSIER_APP/'categories_iris.csv')
exporter_csv(categories_communes,DOSSIER_APP/'categories_communes.csv')
exporter_csv(catalogue,DOSSIER_APP/'catalogue_indicateurs.csv')
exporter_csv(couverture,DOSSIER_APP/'couverture_indicateurs.csv')
exporter_csv(references,DOSSIER_APP/'references_comparaison.csv')
for g,cle_t,nom in [(iris_geo,'IRIS','iris_idf.geojson'),(geo_communes,'CODGEO','communes_idf.geojson')]:
    simple=g[[cle_t,'geometry']].copy()
    simple.geometry=simple.geometry.simplify(SIMPLIFICATION_CARTE_METRES,preserve_topology=True)
    if simple.geometry.is_empty.any() or not simple.geometry.is_valid.all():raise ValueError('Simplification invalide ; réduire le paramètre.')
    exporter_json(json.loads(simple.to_crs(4326).to_json()),DOSSIER_APP/nom)
tailles=pd.DataFrame([{'FICHIER':p.name,'TAILLE_MO':p.stat().st_size/1_000_000} for p in DOSSIER_APP.iterdir()
                     if p.is_file() and p in SORTIES])
exporter_csv(tailles,DOSSIER_INTERIM/'tailles_app_j13e.csv')
if tailles['TAILLE_MO'].gt(MAX_FICHIER_APP_MO).any():raise ValueError('Export applicatif trop lourd ; examiner tailles_app_j13e.csv et réduire la géométrie d’affichage.')
display(tailles)


,FICHIER,TAILLE_MO
0,catalogue_indicateurs.csv,0.005129
1,categories_communes.csv,3.756027
2,categories_iris.csv,13.101323
3,communes_idf.csv,0.364123
4,communes_idf.geojson,9.539488
5,couverture_indicateurs.csv,0.003502
6,iris_idf.csv,2.322185
7,iris_idf.geojson,15.577450
8,points_focus.csv,1.711876
9,references_comparaison.csv,0.004896


## 25 — Dictionnaire, provenance et manifeste
Le dictionnaire utilise la clé MAILLE + COLONNE : un même indicateur peut exister
aux deux mailles sans être un doublon. La provenance reprend les déclarations
amont sans inventer de date de téléchargement. J13e n’ajoute pas de nouvelle source
brute : le fichier sources_a_ajouter_j13e.csv contient uniquement l’en-tête `;`.
Les exports dérivés sont documentés dans fichiers_j13e.csv, pas présentés comme
des téléchargements Insee. Les références techniques restent dans j13e.md.

In [25]:
dictionnaire=catalogue.copy()
dictionnaire['ANNEE_RP_CONTEXTE']=ANNEE_RP;dictionnaire['GEOGRAPHIE_ANALYSE']=GEOGRAPHIE
dictionnaire['MILLESIME_INDICATEUR']='Voir ANNEE_DONNEES et GEOGRAPHIE_SOURCE dans les fiches ; revenus hérités J13b.'
dictionnaire['FORMULE']=dictionnaire['COLONNE'].map({k:f'{f} * {n} / {d}, si {d} > 0' for k,(n,d,f) in FORMULES.items()}).fillna('Reprise du champ publié/contrôlé ou somme complète des effectifs IRIS.')
dictionnaire.loc[dictionnaire['COLONNE'].isin([COL_MED,COL_PAUV]),'FORMULE']='Reprise de l’indicateur fiscal publié et de son statut J13b ; non agrégeable ici.'
dictionnaire.loc[dictionnaire['COLONNE'].eq('J13E_OBS_RESTAURATION_1000_PERSONNES_LIEU_TRAVAIL'),'FORMULE']='1000 * observations retenues / J13D_MOBPRO_LIEU, si lieu > 0 ; dates distinctes.'
dictionnaire['AGREGATION']='Consulter le champ et la maille ; ratios à recalculer sur numérateurs/dénominateurs.'
dictionnaire.loc[dictionnaire['COLONNE'].isin([COL_MED,COL_PAUV]),'AGREGATION']='NON_AGREGABLE_ICI'
exporter_csv(dictionnaire,DOSSIER_PROCESSED/'dictionnaire_diagnostic_j13e.csv')
exporter_csv(dictionnaire,DOSSIER_DOCS/'dictionnaire_j13e.csv')
heritage=[]
for p in FICHIERS_DICTIONNAIRES:
    if p.is_file():
        d=lire_csv(p);d['FICHIER_DICTIONNAIRE']=p.name;heritage.append(d)
if heritage:exporter_csv(pd.concat(heritage,ignore_index=True),DOSSIER_PROCESSED/'dictionnaires_heritage_j13e.csv')
provenance={'j13c':meta_c.get('provenance_sources_declarees',{}),'j13d':meta_d.get('sources',[])}
metadata={'etape':'J13E_DIAGNOSTIC_MARCHE','execution_utc':datetime.now(timezone.utc).isoformat(),
 'annee_rp':ANNEE_RP,'geographie':GEOGRAPHIE,'codes_focus':sorted(FOCUS),'nb_iris':NOMBRE_IRIS,'nb_communes':len(communes),
 'seuils_comparaison':{'population_iris':MIN_POP_IRIS_COMPARAISON,'population_commune':MIN_POP_COMMUNE_COMPARAISON,'n_reference':MIN_REFERENCE},
 'position_relative':'100 * (rang moyen - 1) / (n - 1), ex aequo au rang moyen ; IRIS par type, communes IDF',
 'sources_chargees':SOURCES_CHARGEES,'provenance_amont':provenance,
 'entrees':[{'fichier':str(p),'sha256':empreinte(p)} for p in ENTREES],
 'perimetre_points_app':'IRIS focus affectés OU points valides dans l’emprise rectangulaire du focus + marge',
 'marge_carte_metres':MARGE_CARTE_METRES,'simplification_affichage_metres':SIMPLIFICATION_CARTE_METRES,
 'limites':['Diagnostic descriptif ; pas prévision de fréquentation ou de rentabilité.',
  'Mobilités à la commune seulement ; ne pas sommer le contexte communal entre IRIS.',
  'Les champs travail et études se recoupent ; aucune clientèle unique estimée.',
  'Revenus IRIS conservés avec leurs statuts ; pas de moyenne de médianes.',
  'Comptes géolocalisés retenus ; couverture des affectations à vérifier.',
  'Points IDFM selon unité de la source ; pas nécessairement des gares uniques.',
  'Dates RP, BPE, IDFM et fiscalité distinctes ; pas de photographie d’un seul jour.',
  'Aucun score global ; les positions relatives ne désignent pas un meilleur emplacement.'],
 'logiciels':{'pandas':pd.__version__,'geopandas':gpd.__version__,'plotly':plotly.__version__}}
FICHIER_METADATA=exporter_json(metadata,DOSSIER_PROCESSED/'metadata_diagnostic_j13e.json')
manifest={'etape':'J13E','annee_rp':ANNEE_RP,'geographie':GEOGRAPHIE,'codes_focus':sorted(FOCUS),
 'tables':[{'fichier':r.FICHIER,'taille_mo':r.TAILLE_MO,'sha256':empreinte(DOSSIER_APP/r.FICHIER)} for r in tailles.itertuples()],
 'cles':{'iris_idf.csv':'IRIS','communes_idf.csv':'CODGEO','points_focus.csv':'CLE_POINT'},
 'jointure_contexte':'iris_idf.J13D_COMMUNE_CONTEXTE = communes_idf.CODGEO ; affichage communal, pas répartition IRIS',
 'geometries':'Simplifiées pour affichage ; analyses sur les GPKG complets de processed.',
 'perimetre_points':metadata['perimetre_points_app'],'marge_carte_metres':MARGE_CARTE_METRES,
 'provenance_amont':provenance,'limites':metadata['limites']}
exporter_json(manifest,DOSSIER_APP/'manifest.json')
colonnes_sources=['fichier','source','annee_donnees','annee_geographie','date_telechargement','url']
exporter_csv(pd.DataFrame(columns=colonnes_sources),DOSSIER_DOCS/'sources_a_ajouter_j13e.csv',sep=';')
exporter_csv(pd.DataFrame([{'ROLE':'ENTREE','FICHIER':str(p)} for p in ENTREES]+
                        [{'ROLE':'SORTIE','FICHIER':str(p)} for p in dict.fromkeys(SORTIES)]),DOSSIER_DOCS/'fichiers_j13e.csv')


WindowsPath('C:/Users/almou/OneDrive/GeoMarketing_IDF/docs/fichiers_j13e.csv')

## 26 — Documentation pour GitHub et le client
La documentation décrit la méthode et le contenu destiné à l’application.
Les petits exports data/app/j13e peuvent être versionnés après contrôle de taille
et relecture. Les grandes sources et les HTML embarquant Plotly restent séparés.
Le rapport n’est pas publié sur Internet par ce notebook.

In [26]:
doc=f'''# J13e — Diagnostic territorial et exports applicatifs

## Objectifs
Croiser les profils population, société, fiscalité disponible, offre géolocalisée
et mobilités communales. Conserver {NOMBRE_IRIS} IRIS et {len(communes)} communes.
RP {ANNEE_RP}, géographie {GEOGRAPHIE}, focus {', '.join(sorted(FOCUS))}.

## Entrées
Sorties J13c/J13d : profils CSV, contours complets, points avec leurs statuts,
catégories, métadonnées. Aucune nouvelle source brute. Les notices et dates restent
dans les registres des étapes antérieures et la provenance de J13e.

## Méthode
Lecture stricte des CSV ; contrôle des clés et disponibilité ; sommes complètes
des effectifs IRIS ; recalcul des ratios ; revenus conservés à l’IRIS ; contrôle
des comptes à partir des points ; références IDF (IRIS par type) ; rang moyen
des ex aequo, sans score global. Seuils de population : {MIN_POP_IRIS_COMPARAISON}
pour IRIS, {MIN_POP_COMMUNE_COMPARAISON} pour communes. Référence minimale : {MIN_REFERENCE}.
Les filtres de comparaison ne retirent pas de lignes des profils.

## Sorties
data/processed/j13e : profils complets, focus, catégories, fiches, dictionnaire,
diagnostic long, références et GPKG complets.
data/app/j13e : iris_idf.csv, communes_idf.csv, points_focus.csv, catégories,
catalogue, couverture, références, GeoJSON simplifiés et manifest.json.
outputs/cartes/j13e : 8 cartes/graphiques HTML, sans clé API ni fond OSM.
outputs/rapports/j13e : diagnostic_marche_focus_j13e.html.

## Interprétation
Localisation et comparaison descriptive ; pas fréquentation, potentiel financier
calculé ou score automatique. Les flux ne sont pas mesurés à l’IRIS. Les médianes
fiscales ne sont pas moyennées pour une commune. Les densités dépendent de leur
dénominateur. Les comptes de sources différentes ne donnent pas un nombre unique
d’entreprises ou de clients. La couverture fiscale porte sur des IRIS.

## Application J14
Lire uniquement les exports préparés ; deux niveaux de lecture commune/IRIS ;
filtres, comparaison, cartes, unités et statuts visibles. Les points de l’application
portent seulement sur le focus et sa marge. Les GeoJSON sont pour l’affichage.
J15 : objectif de déploiement après tests locaux et en ligne.

## Références
- https://www.insee.fr/fr/metadonnees/definition/c1523
- https://www.insee.fr/fr/statistiques/8590191
- https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.dissolve.html
- https://plotly.com/python/choropleth-maps/
- https://nbconvert.readthedocs.io/en/latest/config_options.html
'''
FICHIER_DOC=DOSSIER_DOCS/'j13e.md'
FICHIER_DOC.write_text(doc,encoding='utf-8');SORTIES.append(FICHIER_DOC)
print('Documentation :',FICHIER_DOC)


Documentation : C:\Users\almou\OneDrive\GeoMarketing_IDF\docs\j13e.md


## 27 — Contrôles finaux avant interprétation
Ne préparer le commit qu’après ces contrôles. Les valeurs et statuts hérités
sont comparés à l’entrée. Les contrôles portent aussi sur les ex aequo, les parts,
les valeurs manquantes et la conservation de la maille communale des flux.

In [27]:
assert len(profil_iris)==NOMBRE_IRIS and set(profil_iris['IRIS'])==ATTENDUS and profil_iris['IRIS'].is_unique
pd.testing.assert_frame_equal(profil_iris.set_index('IRIS')[base.columns.drop('IRIS')],
                              base.set_index('IRIS')[base.columns.drop('IRIS')],check_dtype=True)
pd.testing.assert_frame_equal(communes.set_index('CODGEO')[communes_amont.columns.drop('CODGEO')],
                              communes_amont.set_index('CODGEO')[communes_amont.columns.drop('CODGEO')],check_dtype=True)
assert set(app_iris['IRIS'])==ATTENDUS and app_iris['IRIS'].is_unique
assert set(app_communes['CODGEO'])==set(communes_amont['CODGEO']) and app_communes['CODGEO'].is_unique
assert not any(c.startswith('J13D_CONTEXTE_COMMUNAL_') for c in app_iris)
assert not dictionnaire.duplicated(['MAILLE','COLONNE']).any()
for t in [profil_iris,communes]:
    for col in FORMULES:
        n=nombre(t[col],col)
        assert n.dropna().ge(0).all()
        if col.endswith('_PCT'):assert n.dropna().le(100+1e-6).all()
    for col in [c for c in t if c.startswith('J13E_POSITION_')]:assert t[col].dropna().between(0,100).all()
for t,cle_t in [(categories_iris,'IRIS'),(categories_communes,'CODGEO')]:
    if not t.empty:assert np.allclose(t.groupby([cle_t,'SOURCE'])['PART_DANS_SOURCE_PCT'].sum(),100)
assert len(FICHIERS_HTML)==8 and all(p.is_file() for p in FICHIERS_HTML)
assert all(p.is_file() for p in SORTIES)
for r in metadata['entrees']:assert empreinte(r['fichier'])==r['sha256']
controle=pd.DataFrame([{'CONTROLE':n,'ETAT':'VALIDE'} for n in [
 'IRIS et communes conservés','Colonnes et statuts amont conservés','Points et comptes cohérents',
 'Catégories et parts cohérentes','Ratios et positions contrôlés','Exports présents','Empreintes des entrées inchangées']])
exporter_csv(controle,DOSSIER_INTERIM/'controle_final_j13e.csv')
display(controle)
print('J13e terminé ; lire le rapport et les diagnostics avant diffusion.')


,CONTROLE,ETAT
0,IRIS et communes conservés,VALIDE
1,Colonnes et statuts amont conservés,VALIDE
2,Points et comptes cohérents,VALIDE
3,Catégories et parts cohérentes,VALIDE
4,Ratios et positions contrôlés,VALIDE
5,Exports présents,VALIDE
6,Empreintes des entrées inchangées,VALIDE


J13e terminé ; lire le rapport et les diagnostics avant diffusion.


## 29 — Suite : J14, première application Streamlit
Le prochain notebook ou script n’a plus à relire les grandes sources nationales.
Créer app.py et des modules de lecture à partir de data/app/j13e/ :
1. Vue commune : population, offre retenue et contexte travail/études.
2. Vue IRIS : population, profils, revenus disponibles et offre locale.
3. Comparaison : valeurs, unités, couverture et référence clairement indiquées.
4. Carte focus : contours et points avec noms uniquement au survol.
5. Méthode : sources, dates, disponibilités et limites.
L’interface joint le contexte communal à la consultation d’un IRIS par
J13D_COMMUNE_CONTEXTE, sans le présenter comme une mesure de quartier. Le catalogue
permet d’afficher des libellés lisibles. Les valeurs manquantes sont affichées
« Non disponible » ; la présence de zéro n’est pas un statut de source absente.
J15 reste l’objectif de premier déploiement après tests. Une étude client peut
ensuite intégrer les visites terrain, loyers, horaires, prix, passages observés,
campus et temps de marche. Ces informations ne sont pas déduites des flux communaux.